In [3]:
import pandas as pd
import numpy as np
import pyodbc
import os

In [39]:
df = pd.read_csv('Source Files\\batch_01_file_01.csv' , sep='|')
df

,id,imsi,imei,cell,lac,event_type,event_ts
0,1,257865245.0,1.806594e+13,6226.0,75,5.0,20/12/2020 01:25
1,2,NaN,5.077664e+13,5234.0,80,4.0,16/12/2020 02:17
2,3,248622693.0,3.106621e+13,4755.0,33,9.0,27/12/2020 05:06
3,4,866701003.0,6.133836e+13,2814.0,48,2.0,30/12/2020 15:47
4,5,570476314.0,3.209416e+13,3944.0,24,NaN,12/12/2020 10:04
5,6,314240287.0,1.233567e+13,5757.0,56,6.0,11/12/2020 09:49
6,7,305086458.0,2.075140e+13,1967.0,78,1.0,27/12/2020 19:57
7,8,719398339.0,6.703115e+13,NaN,76,4.0,30/12/2020 19:54
8,9,856150031.0,9.859162e+13,1870.0,15,5.0,25/12/2020 03:32
9,10,419999500.0,4.507790e+13,2309.0,31,8.0,29/12/2020 13:34


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          20 non-null     int64  
 1   imsi        19 non-null     float64
 2   imei        19 non-null     float64
 3   cell        19 non-null     float64
 4   lac         20 non-null     int64  
 5   event_type  19 non-null     float64
 6   event_ts    19 non-null     object 
dtypes: float64(4), int64(2), object(1)
memory usage: 1.2+ KB


In [41]:
df.dropna(subset=['imsi' , 'cell' , 'lac' , 'event_type' , 'event_ts'], inplace=True)
df

,id,imsi,imei,cell,lac,event_type,event_ts
0,1,257865245.0,1.806594e+13,6226.0,75,5.0,20/12/2020 01:25
2,3,248622693.0,3.106621e+13,4755.0,33,9.0,27/12/2020 05:06
3,4,866701003.0,6.133836e+13,2814.0,48,2.0,30/12/2020 15:47
5,6,314240287.0,1.233567e+13,5757.0,56,6.0,11/12/2020 09:49
6,7,305086458.0,2.075140e+13,1967.0,78,1.0,27/12/2020 19:57
8,9,856150031.0,9.859162e+13,1870.0,15,5.0,25/12/2020 03:32
9,10,419999500.0,4.507790e+13,2309.0,31,8.0,29/12/2020 13:34
11,12,349932947.0,5.687233e+13,4487.0,87,6.0,08/12/2020 07:22
12,13,889798900.0,7.357449e+13,2794.0,95,8.0,01/12/2020 19:25
13,14,303061908.0,NaN,6781.0,37,7.0,11/12/2020 16:30


In [42]:
df['imsi'] = df['imsi'].astype(str).str.split('.').str[0]
df['cell'] = df['cell'].astype(int)
df['lac'] = df['lac'].astype(int)
df['event_type'] = df['event_type'].astype(str).str.split('.').str[0]
df['imei'] = df['imei'].astype(int,errors= 'ignore')
df['imei'] = df['imei'].astype(str,errors= 'ignore').str.split('.').str[0]
df['event_ts'] = pd.to_datetime(df['event_ts'])
df

C:\Users\faster\AppData\Local\Temp\ipykernel_21640\1065107106.py:7: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['event_ts'] = pd.to_datetime(df['event_ts'])


,id,imsi,imei,cell,lac,event_type,event_ts
0,1,257865245,18065935567874,6226,75,5,2020-12-20 01:25:00
2,3,248622693,31066209199051,4755,33,9,2020-12-27 05:06:00
3,4,866701003,61338362724146,2814,48,2,2020-12-30 15:47:00
5,6,314240287,12335667771826,5757,56,6,2020-12-11 09:49:00
6,7,305086458,20751395545290,1967,78,1,2020-12-27 19:57:00
8,9,856150031,98591624192001,1870,15,5,2020-12-25 03:32:00
9,10,419999500,45077904630108,2309,31,8,2020-12-29 13:34:00
11,12,349932947,56872332696514,4487,87,6,2020-12-08 07:22:00
12,13,889798900,73574489815827,2794,95,8,2020-12-01 19:25:00
13,14,303061908,nan,6781,37,7,2020-12-11 16:30:00


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 0 to 19
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          16 non-null     int64         
 1   imsi        16 non-null     object        
 2   imei        16 non-null     object        
 3   cell        16 non-null     int64         
 4   lac         16 non-null     int64         
 5   event_type  16 non-null     object        
 6   event_ts    16 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 1.0+ KB


In [44]:
conn = pyodbc.connect(                          ## Declare connection
    "driver={ODBC Driver 17 for SQL Server};"
    "server=.\\SQLSERVER;"
    "database=SSIS_Telecom_DB;"
    "trusted_connection=yes;"
)

cursor = conn.cursor()

In [45]:
refrence = pd.read_sql_query('select * from dim_imsi_reference', conn)

C:\Users\faster\AppData\Local\Temp\ipykernel_21640\769725347.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  refrence = pd.read_sql_query('select * from dim_imsi_reference', conn)


In [46]:
refrence

,id,imsi,subscriber_id
0,1,919106573,62306
1,2,835424796,12831
2,3,222893094,14084
3,4,461051324,74624
4,5,720757084,21855
...,...,...,...
995,996,752043376,17864
996,997,811085551,56837
997,998,871385869,80010
998,999,509980545,27726


In [47]:
pre_Final = df.merge(refrence, how='left', on='imsi')

In [48]:
pre_Final

,id_x,imsi,imei,cell,lac,event_type,event_ts,id_y,subscriber_id
0,1,257865245,18065935567874,6226,75,5,2020-12-20 01:25:00,174,78663
1,3,248622693,31066209199051,4755,33,9,2020-12-27 05:06:00,175,84988
2,4,866701003,61338362724146,2814,48,2,2020-12-30 15:47:00,176,19661
3,6,314240287,12335667771826,5757,56,6,2020-12-11 09:49:00,178,78908
4,7,305086458,20751395545290,1967,78,1,2020-12-27 19:57:00,179,41694
5,9,856150031,98591624192001,1870,15,5,2020-12-25 03:32:00,180,52042
6,10,419999500,45077904630108,2309,31,8,2020-12-29 13:34:00,181,67086
7,12,349932947,56872332696514,4487,87,6,2020-12-08 07:22:00,182,87249
8,13,889798900,73574489815827,2794,95,8,2020-12-01 19:25:00,183,52066
9,14,303061908,nan,6781,37,7,2020-12-11 16:30:00,184,28821


In [49]:
pre_Final.columns

Index(['id_x', 'imsi', 'imei', 'cell', 'lac', 'event_type', 'event_ts', 'id_y',
       'subscriber_id'],
      dtype='object')

In [50]:
pre_Final['imei'] = pre_Final['imei'].astype('string')

invalid_imei_mask = pre_Final['imei'].isna() | (pre_Final['imei'].str.len() < 14)

pre_Final['TAC'] = pre_Final['imei'].str[:8]
pre_Final['SNR'] = pre_Final['imei'].str[-6:]

pre_Final.loc[invalid_imei_mask, 'TAC'] = '-99999'
pre_Final.loc[invalid_imei_mask, 'SNR'] = '-99999'
pre_Final.loc[invalid_imei_mask, 'imei'] = '-99999'

pre_Final['TAC'] = pre_Final['TAC'].fillna('-99999')
pre_Final['SNR'] = pre_Final['SNR'].fillna('-99999')
pre_Final['imei'] = pre_Final['imei'].fillna('-99999')

pre_Final['subscriber_id'] = pre_Final['subscriber_id'].fillna(-99999)
pre_Final['subscriber_id'] = pre_Final['subscriber_id'].astype(int)

In [51]:
pre_Final

,id_x,imsi,imei,cell,lac,event_type,event_ts,id_y,subscriber_id,TAC,SNR
0,1,257865245,18065935567874,6226,75,5,2020-12-20 01:25:00,174,78663,18065935,567874
1,3,248622693,31066209199051,4755,33,9,2020-12-27 05:06:00,175,84988,31066209,199051
2,4,866701003,61338362724146,2814,48,2,2020-12-30 15:47:00,176,19661,61338362,724146
3,6,314240287,12335667771826,5757,56,6,2020-12-11 09:49:00,178,78908,12335667,771826
4,7,305086458,20751395545290,1967,78,1,2020-12-27 19:57:00,179,41694,20751395,545290
5,9,856150031,98591624192001,1870,15,5,2020-12-25 03:32:00,180,52042,98591624,192001
6,10,419999500,45077904630108,2309,31,8,2020-12-29 13:34:00,181,67086,45077904,630108
7,12,349932947,56872332696514,4487,87,6,2020-12-08 07:22:00,182,87249,56872332,696514
8,13,889798900,73574489815827,2794,95,8,2020-12-01 19:25:00,183,52066,73574489,815827
9,14,303061908,-99999,6781,37,7,2020-12-11 16:30:00,184,28821,-99999,-99999


In [52]:
pre_Final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_x           16 non-null     int64         
 1   imsi           16 non-null     object        
 2   imei           16 non-null     string        
 3   cell           16 non-null     int64         
 4   lac            16 non-null     int64         
 5   event_type     16 non-null     object        
 6   event_ts       16 non-null     datetime64[ns]
 7   id_y           16 non-null     int64         
 8   subscriber_id  16 non-null     int64         
 9   TAC            16 non-null     string        
 10  SNR            16 non-null     string        
dtypes: datetime64[ns](1), int64(5), object(2), string(3)
memory usage: 1.5+ KB


In [53]:
pre_Final.drop(columns=['id_y'], inplace=True)
pre_Final.rename(columns={'id_x' : 'Transaction_id'}, inplace=True)

In [54]:
pre_Final

,Transaction_id,imsi,imei,cell,lac,event_type,event_ts,subscriber_id,TAC,SNR
0,1,257865245,18065935567874,6226,75,5,2020-12-20 01:25:00,78663,18065935,567874
1,3,248622693,31066209199051,4755,33,9,2020-12-27 05:06:00,84988,31066209,199051
2,4,866701003,61338362724146,2814,48,2,2020-12-30 15:47:00,19661,61338362,724146
3,6,314240287,12335667771826,5757,56,6,2020-12-11 09:49:00,78908,12335667,771826
4,7,305086458,20751395545290,1967,78,1,2020-12-27 19:57:00,41694,20751395,545290
5,9,856150031,98591624192001,1870,15,5,2020-12-25 03:32:00,52042,98591624,192001
6,10,419999500,45077904630108,2309,31,8,2020-12-29 13:34:00,67086,45077904,630108
7,12,349932947,56872332696514,4487,87,6,2020-12-08 07:22:00,87249,56872332,696514
8,13,889798900,73574489815827,2794,95,8,2020-12-01 19:25:00,52066,73574489,815827
9,14,303061908,-99999,6781,37,7,2020-12-11 16:30:00,28821,-99999,-99999


In [55]:
Final = pd.DataFrame(pre_Final[['Transaction_id' , 'imsi' , 'subscriber_id' ,
                                'TAC' , 'SNR' , 'imei' , 'cell' , 'lac' , 'event_type' , 'event_ts' ,]])

In [56]:
Final

,Transaction_id,imsi,subscriber_id,TAC,SNR,imei,cell,lac,event_type,event_ts
0,1,257865245,78663,18065935,567874,18065935567874,6226,75,5,2020-12-20 01:25:00
1,3,248622693,84988,31066209,199051,31066209199051,4755,33,9,2020-12-27 05:06:00
2,4,866701003,19661,61338362,724146,61338362724146,2814,48,2,2020-12-30 15:47:00
3,6,314240287,78908,12335667,771826,12335667771826,5757,56,6,2020-12-11 09:49:00
4,7,305086458,41694,20751395,545290,20751395545290,1967,78,1,2020-12-27 19:57:00
5,9,856150031,52042,98591624,192001,98591624192001,1870,15,5,2020-12-25 03:32:00
6,10,419999500,67086,45077904,630108,45077904630108,2309,31,8,2020-12-29 13:34:00
7,12,349932947,87249,56872332,696514,56872332696514,4487,87,6,2020-12-08 07:22:00
8,13,889798900,52066,73574489,815827,73574489815827,2794,95,8,2020-12-01 19:25:00
9,14,303061908,28821,-99999,-99999,-99999,6781,37,7,2020-12-11 16:30:00


In [57]:
Final.to_csv("Processed Files\\batch_01_file_01.csv", index=False)